# 意图识别 — 基线方案 (baseline)

这是一个**最简提交**示例，用来给参赛选手提供一个明确的下限基线：

- 直接在 `datasets/train.jsonl`（16 行，每类 1 条）上微调 `bert-base-chinese`
- 最终生成 `submission.zip` (内含 `submission_val.jsonl` 与 `submission_test.jsonl`)

**注意**: 16 行训练数据 × 3 epochs 在 16 batch_size 下只有 ~3 个梯度步，BERT 远未收敛，分数会很低。本基线只为提交格式与流程做演示。

In [ ]:
from __future__ import annotations

import json
import random
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm
import random

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE = {DEVICE}")
PATH = '/bohr/train-z7v5/v1/'
BASE_MODEL_NAME = "bert-base-chinese"
# MODEL_DIR = PATH + "models--bert-base-chinese/8f23c25b06e129b6c986331a13d8d025a92cf0ea"
MODEL_DIR = PATH + "bert-base-chinese"
TRAIN_PATH = Path(PATH + "/train.jsonl")


MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 2e-5
SEED = 42


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


set_seed(SEED)

In [ ]:
INTENT_LABELS = [
    "游戏技巧", "游戏角色信息", "周边饭馆", "查找高端酒店", "查找地址",
    "健康知识", "查找医疗信息", "美容化妆技巧", "美食烹饪技巧", "软件开发问题",
    "软件使用问题", "查找小说剧情", "查找小说角色信息", "查找营业时间",
    "法律法条解释", "法律问题咨询",
]
LABEL_TO_ID = {l: i for i, l in enumerate(INTENT_LABELS)}
ID_TO_LABEL = {i: l for i, l in enumerate(INTENT_LABELS)}

In [ ]:
def extract_user_utterance(text: str) -> str:
    user_utts = []
    for line in text.strip().split("\n"):
        if line.strip().startswith("usr:"):
            user_utts.append(line.strip()[4:].strip())
    return user_utts[-1] if user_utts else ""


def preprocess_input_text(text: Any) -> str:
    if text is None:
        return ""
    s = str(text).strip()
    if not s:
        return ""
    if "usr:" in s:
        u = extract_user_utterance(s)
        if u:
            return u
    return s


def load_jsonl(path):
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


train_records = load_jsonl(TRAIN_PATH)
train_rows = []
for r in train_records:
    text = preprocess_input_text(r.get("input"))
    lab = str(r.get("label", "")).strip()
    if text and lab in LABEL_TO_ID:
        train_rows.append({"text": text, "label_id": LABEL_TO_ID[lab]})
print(f"训练样本: {len(train_rows)}")

In [ ]:
from transformers import BertTokenizer, BertModel
class IntentDataset(Dataset):
    def __init__(self, rows, tokenizer, max_len, with_label=True):
        self.rows = rows; self.tokenizer = tokenizer; self.max_len = max_len; self.with_label = with_label

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        enc = self.tokenizer(row["text"], add_special_tokens=True, truncation=True,
                             max_length=self.max_len, padding="max_length", return_tensors="pt")
        item = {"input_ids": enc["input_ids"].flatten(), "attention_mask": enc["attention_mask"].flatten()}
        if self.with_label:
            item["labels"] = torch.tensor(row["label_id"], dtype=torch.long)
        return item
    

class Nueryim_TS(Dataset):
    def __init__(self, file, labl, tokenizer, max_len):
        self.data = []
        with open(file, 'r', encoding='utf-8') as f:
            for line in f:
                self.data.append(line)
        self.labl = torch.tensor(labl, dtype=torch.long)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return  len(self. data)
    def __getitem__(self, indx):
        line = self.data[indx]
        encd = self.tokenizer(line, add_special_tokens=True, truncation=True,
                             max_length=self.max_len, padding="max_length", return_tensors="pt")
        return {
            "input_ids":        encd["input_ids"].flatten(),
            "attention_mask":   encd["attention_mask"].flatten(),
            'labels': self.labl,
        }

class BertIntentClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        if isinstance(out, tuple):
            pooled = out[0][:, 0]
        else:
            pooled = getattr(out, "pooler_output", None)
            if pooled is None:
                pooled = out.last_hidden_state[:, 0]
        return self.classifier(self.dropout(pooled))


tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)
model = BertIntentClassifier(MODEL_DIR, len(INTENT_LABELS)).to(DEVICE)

DATA = Path('/bohr/dgpt-61fx/v1/data_GPT')

s_train = [IntentDataset(train_rows, tokenizer, MAX_LEN)]
for indx in range(16):
    s_train.append(Nueryim_TS(DATA / f'after_{indx+1}.txt', indx, tokenizer, MAX_LEN))
s_train = ConcatDataset(s_train)

train_loader = DataLoader(s_train, batch_size=BATCH_SIZE, shuffle=True)
optim = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = max(1, len(train_loader) * EPOCHS)
warmup_steps = max(1, int(total_steps * 0.1))
sched = get_linear_schedule_with_warmup(optim, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

for ep in range(EPOCHS):
    print(f"epoch {ep+1}/{EPOCHS}")
    model.train()
    for batch in tqdm(train_loader, desc="train", leave=False):
        optim.zero_grad()
        x = batch["input_ids"].to(DEVICE); m = batch["attention_mask"].to(DEVICE); y = batch["labels"].to(DEVICE)
        loss = nn.CrossEntropyLoss()(model(x, m), y)
        loss.backward(); optim.step(); sched.step()
print("training done.")

In [ ]:
import os

if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"  
else:
    print("Baseline运行时，因为无法读取测试集，所以会有此条报错，属于正常现象")  
    print("When baseline is running, this error message will appear because the test set cannot be read, which is a normal phenomenon.")

TEST_A_PATH = Path(DATA_PATH + "/val.jsonl")
TEST_B_PATH = Path(DATA_PATH + "/test.jsonl")


@torch.no_grad()
def predict_to_file(test_path, out_path):
    model.eval()
    records = load_jsonl(test_path)
    rows = [{"text": preprocess_input_text(r.get("input")) or ""} for r in records]
    loader = DataLoader(IntentDataset(rows, tokenizer, MAX_LEN, with_label=False), batch_size=BATCH_SIZE*4, shuffle=False)
    preds = []
    for batch in tqdm(loader, desc="predict", leave=False):
        x = batch["input_ids"].to(DEVICE); m = batch["attention_mask"].to(DEVICE)
        preds.extend(torch.argmax(model(x, m), dim=1).cpu().tolist())
    assert len(preds) == len(records)
    with Path(out_path).open("w", encoding="utf-8") as fh:
        for p in preds:
            fh.write(json.dumps({"label": ID_TO_LABEL[p]}, ensure_ascii=False) + "\n")
    print(f"wrote {len(preds)} -> {out_path}")


predict_to_file(TEST_A_PATH, Path("submission_val.jsonl"))
predict_to_file(TEST_B_PATH, Path("submission_test.jsonl"))

In [ ]:
SUBMISSION_ZIP = Path("submission.zip")
with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("submission_val.jsonl")
    zf.write("submission_test.jsonl")
print(f"wrote {SUBMISSION_ZIP} ({SUBMISSION_ZIP.stat().st_size} bytes)")

In [ ]:
'''

import os
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 扩展用户主目录路径
model_path = os.path.expanduser("~/Qwen3-4B-Instruct")

# 检测可用设备
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"正在使用设备: {device}")

# 加载分词器和模型（使用自动推断的精度，若GPU可用则自动使用bfloat16）
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

# 定义对话消息（遵循Qwen的chat模板）
messages = [
    {"role": "system", "content": "你是一个有用的助手。"},
    {"role": "user", "content": "你好，请介绍一下你自己。"}
]

# 应用聊天模板生成输入
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 对输入进行编码并移至模型设备
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# 生成回复参数（可根据需要调整）
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    top_p=0.8,
    repetition_penalty=1.05
)

# 仅提取生成的部分（去除原始输入）
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# 解码回复
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("\n模型回复：")
print(response)
'''